In [1]:
import pickle
import os
import pandas as pd

In [2]:
res_discomat = pickle.load(open('comp_res_gold.pkl', 'rb'))

In [3]:
print(res_discomat.keys())
# print(res_discomat['pred_row_col_labels'][0])
# print(res_discomat['pred_tuples'][:5])

dict_keys(['identifier', 'pred_tuples'])


In [4]:
prop_res = pickle.load(open('prop_res_gold.pkl', 'rb'))

In [5]:
print(prop_res.keys())

dict_keys(['identifier', 'ret_comp_pred', 'pred_tuples'])


In [6]:
prop_res['test'] = prop_res
# print(prop_res['test'].keys())
# print(prop_res['test']['identifier'][:5])
# print(prop_res['test']['identifier'][:5])
# print(prop_res['test']['ret_comp_pred'][:5])
pred_prop_tuples_test = prop_res['test']['pred_tuples']
# print(len(pred_prop_tuples_test))
# print(pred_prop_tuples_test[0], pred_prop_tuples_test[500])

In [7]:
fpred_disco_tuples = res_discomat['pred_tuples']
pred_disco_tuples = []
for tup in fpred_disco_tuples:
    if any(tup):
        pred_disco_tuples += tup
pred_disco_tuples_test = []
for tup in pred_disco_tuples:
    #removing blank tuples
    if any(tup):
        pred_disco_tuples_test.append(tup)

In [8]:
print(len(pred_disco_tuples_test))

9540


In [9]:
print(pred_disco_tuples_test[0:5]) 
print(pred_disco_tuples_test[180:185])

[('S0022309304006957_0_2_1_0_1', 'PbO', 30.0, 'mol'), ('S0022309304006957_0_2_1_0_1', 'SiO2', 70.0, 'mol'), ('S0022309304006957_0_2_2_0_2', 'PbO', 33.0, 'mol'), ('S0022309304006957_0_2_2_0_2', 'SiO2', 67.0, 'mol'), ('S0022309304006957_0_2_3_0_3', 'PbO', 43.0, 'mol')]
[('S0022309303007567_2_10_3_0', 'P', 0.10514, 'mol'), ('S0022309303007567_2_10_3_0', 'S', 0.632, 'mol'), ('S0022309303007567_2_10_4_0', 'Ge', 0.27786, 'mol'), ('S0022309303007567_2_10_4_0', 'P', 0.11114, 'mol'), ('S0022309303007567_2_10_4_0', 'S', 0.611, 'mol')]


In [10]:
#assiging orientation to props
for i in range(len(prop_res['test']['ret_comp_pred'])):
    count_col, count_row = 0, 0
    row_label = prop_res['test']['ret_comp_pred'][i]['pred_row_label']
    col_label = prop_res['test']['ret_comp_pred'][i]['pred_col_label']
    #print(row_col)
    prop_list = list(range(2,21))
    prop_res['test']['ret_comp_pred'][i]['prop_orient'] = None
    count_col = sum(1 for element in col_label if element >= 2)
    count_row = sum(1 for element in row_label if element >= 2)
    if count_col>= count_row:
        prop_res['test']['ret_comp_pred'][i]['prop_orient'] = 'col'
    else:
        prop_res['test']['ret_comp_pred'][i]['prop_orient'] = 'row'
        
#     if count_col and count_row:
#         print(prop_res['test']['identifier'][i])
#         print(prop_res['test']['ret_comp_pred'][i])
#         print()

In [11]:
def find_occ(ini_str, sub_str, occ):
    val = -1
    for i in range(0, occ):
        val = ini_str.find(sub_str, val + 1)
    return val

In [12]:
# modifying id to connect property and composition
def find_gid_prop(prop, orient):
    if orient == 'col':
        locp = find_occ(prop[0], '_', 3)
        gid_prop = prop[0][:locp]
        return gid_prop
        
    elif orient == 'row':
        locpe = find_occ(prop[0], '_', 3)
        locpi = find_occ(prop[0], '_', 2)
        gid_prop = prop[0][:locpi] + '_' + prop[0][locpe + 1]
        return gid_prop
        
    else: raise Exception(f'No orientation assigned for {pii_tid}')

In [13]:
def conn_pred_tuples_no_id(pii_tid_list):
    
    '''
    Code to connect composition and property according to orientation without any id
    only for intra table 
    
    '''
    id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set, matched_id_set = [], [], [], [], [], set(), set()
    prop_dict = {'prop_2':'d', 'prop_3':'tg', 'prop_4':'ri', 'prop_5':'abbe'}
    for pii_tid in pii_tid_list:
#         print(f'pii_tid = {pii_tid}')
        ind = prop_res['test']['identifier'].index(pii_tid)
#         pred_prop_tuples =prop_res['test']['ret_tuples_pred'][ind]
        pred_prop_tuples = result_dict[pii_tid]
        #print(len(pred_prop_tuples))
        #print(pred_prop_tuples)
        inde = res_discomat['identifier'].index(pii_tid)
        pred_comp_tuples = res_discomat['pred_tuples'][inde]
        print(pii_tid)
        print(pred_comp_tuples)
        print(pred_prop_tuples)
        print()
        #print(pred_comp_tuples)
        #print()

        
        prop_orient = prop_res['test']['ret_comp_pred'][ind]['prop_orient']
        for comp in pred_comp_tuples:
            gid_comp = find_gid_prop(comp, prop_orient)
#             print(comp)
#             print(gid_comp)
#             print()
            
            for prop in pred_prop_tuples:
                gid_prop = find_gid_prop(prop, prop_orient)
#                 print(prop)
#                 print(gid_prop)
#                 print()
                if gid_comp == gid_prop:
                    matched_id_set.add(pii_tid)
                    if gid_comp not in id_list:
                        #print(f'New id == {gid_comp}')
                        id_list.append(gid_comp)
                        
                        pparts = prop[0].split('_')
                        org_id = ''
                        if len(pparts)==5:
                            org_id = pparts[-1]
                        orig_id_list.append(org_id)
                        
                        proxy_id = gid_prop.split('_')[-1]
                        if prop_orient == 'col':
                            proxy_idf = pii_tid[0] + '_' + str(pii_tid[1]) + '_' + 'R' + '_' + str(int(proxy_id) + 1) 
                        elif prop_orient == 'row':
                            proxy_idf = pii_tid[0] + '_' + str(pii_tid[1]) + '_' + 'C' + '_' + str(int(proxy_id) + 1) 
                        proxy_id_list.append(proxy_idf)
                        
                        comp_list.append([])
                        prop_list.append([])
                    insert_ind = id_list.index(gid_comp)
                    #print(insert_ind)
                    if comp[1:] not in comp_list[insert_ind]:
                        comp_list[insert_ind].append(comp[1:])
                    pr_n_list = list(prop)
                    #pr_n_list[1] = prop_dict[pr_n_list[1]]
                    prop = tuple(pr_n_list)
                    if prop[1:] not in prop_list[insert_ind]:
                        prop_list[insert_ind].append(prop[1:])
#                         print(f'{comp} ==  {prop}')
                else:
                    if gid_comp[:gid_comp.rindex('_')] != gid_prop[:gid_prop.rindex('_')]:
                        mismatch_id_set.add(pii_tid)
                    #mismatch_id_set.add(pii_tid)
            
#     for i in range(len(id_list)):
#         print(f'{id_list[i]} -> {orig_id_list} -> {proxy_id_list[i]} -> {comp_list[i]} -> {prop_list[i]}')
        #print(f'{comp_list[i]} ---> {prop_list[i]}')
        
    assert len(id_list) == len(orig_id_list) == len(proxy_id_list) == len(comp_list) == len(prop_list), 'Comp and prop length diff, not possible'
    
#     print(f'No of materials present = {len(comp_list)}')
#     print(f'No of matched ids in true matching = {len(matched_id_set)}')
    mismatch_id_set = set(pii_tid_list) - matched_id_set
#     print(f'No of mismatched ids in pred matching = {len(mismatch_id_set)}')
    #print(f'Mismatching ids == {mismatch_id_set}')
    return id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set

In [14]:
# pred_prop_tuples_test

In [15]:
### make 3 lists for pred, comp_prop, comp, prop

pred_comp_prop_pii, pred_comp_pii, pred_prop_pii = [], [], []

#make comp list for test
for tup in pred_disco_tuples_test:
    if any(tup):
#         print(tup)
        uned_pii_tid = tup[0]
        parts = uned_pii_tid.split('_')
        pii_tid = (parts[0], int(parts[1]))
        if pii_tid not in pred_comp_pii:
            pred_comp_pii.append(pii_tid)
    
#print(pred_comp_pii[:5])

#make comp list for test
for tup in pred_prop_tuples_test:
    if any(tup):
#         print(tup)
        uned_pii_tid = tup[0]
        parts = uned_pii_tid.split('_')
        pii_tid = (parts[0], int(parts[1]))
        if pii_tid not in pred_prop_pii:
            pred_prop_pii.append(pii_tid)
    
#print(pred_prop_pii[:5])

pred_comp_prop_pii = list(set(pred_comp_pii) & set(pred_prop_pii)) 
pred_only_comp_pii = list(set(pred_comp_pii) - set(pred_comp_prop_pii))
pred_only_prop_pii = list(set(pred_prop_pii) - set(pred_comp_prop_pii))
print(len(pred_comp_prop_pii), len(pred_only_comp_pii), len(pred_only_prop_pii))
print(pred_comp_prop_pii[0], pred_only_comp_pii[0], pred_only_prop_pii[0])

106 252 83
('S0022309309001586', 0) ('S0022309314003688', 1) ('S0022309302015764', 1)


In [16]:
from collections import defaultdict


result_dict = defaultdict(list)
pred_tuples = prop_res['test']['pred_tuples']
for tup in pred_tuples:
#     print(tup)
    # Split the first element of the tuple to extract the key
    key_parts = tup[0].split('_')
    key = (key_parts[0], int(key_parts[1]))
    result_dict[key].append(tup)

In [17]:
# result_dict[('S0022309303007567', 0)]

In [18]:
id_list, orig_id_list, proxy_id_list, comp_list, prop_list, mismatch_id_set = conn_pred_tuples_no_id(pred_comp_prop_pii)

('S0022309309001586', 0)
[('S0022309309001586_0_2_1_0_L1', 'La', 25.0, 'mol'), ('S0022309309001586_0_2_1_0_L1', 'P', 75.0, 'mol'), ('S0022309309001586_0_2_1_0_L1', 'La', 24.51, 'mol'), ('S0022309309001586_0_2_1_0_L1', 'P', 75.49, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'La', 20.0, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'Ca', 5.0, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'P', 75.0, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'La', 18.56, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'Ca', 5.26, 'mol'), ('S0022309309001586_0_3_1_0_C1', 'P', 76.18, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'La', 15.0, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'Ca', 10.0, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'P', 75.0, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'La', 13.68, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'Ca', 10.12, 'mol'), ('S0022309309001586_0_4_1_0_C2', 'P', 76.21, 'mol'), ('S0022309309001586_0_5_1_0_C3', 'La', 10.0, 'mol'), ('S0022309309001586_0_5_1_0_C3', 'Ca', 15.0, 'mol'), ('S0022309309001586_

In [19]:
print(len(mismatch_id_set), len(id_list))

6 731


In [20]:
def sort_comp(single_list_comp):
    '''
    for sorting  ordering of chemical components alphabetically
    '''
    single_list_comp_sorted = sorted(single_list_comp, key=lambda x: x[0])
    return single_list_comp_sorted

In [21]:
new_list_final = []
print(id_list)
for index in range(len(id_list)):
    for prop_single in prop_list[index]:
        parts = id_list[index].split('_')
        output_string = '_'.join(parts[:2])
        my_id = orig_id_list[index]
        sorted_comp = sort_comp(comp_list[index]) # sortin existing tuples
        new_list_final.append([parts[0], [int(parts[1])+1], my_id, proxy_id_list[index], sorted_comp, prop_single])
        
print(new_list_final[:5])
print(len(new_list_final))
print(new_list_final[0])
print(len(new_list_final[0]))
start_search = len(new_list_final)

['S0022309309001586_0_2', 'S0022309309001586_0_3', 'S0022309309001586_0_4', 'S0022309309001586_0_5', 'S0022309309001586_0_6', 'S0022309309001586_0_7', 'S0022309309001586_0_8', 'S0022309309001586_0_9', 'S0022309309001586_0_10', 'S0022309309001586_0_11', 'S0022309309001586_0_12', 'S0022309309001586_0_13', 'S0022309309001586_0_14', 'S0022309301007268_2_1', 'S0022309301007268_2_3', 'S0022309301007268_2_4', 'S0022309301007268_2_5', 'S0022309301007268_2_6', 'S0022309301007268_2_8', 'S0022309301007268_2_9', 'S0022309301007268_2_10', 'S0022309301007268_2_11', 'S0022309304006969_1_1', 'S0022309304006969_1_2', 'S0022309304006969_1_3', 'S0022309304006969_1_4', 'S0022309304006969_1_5', 'S0022309304006969_1_6', 'S0022309304006969_1_7', 'S0022309304006969_1_8', 'S0022309307001524_0_3', 'S0022309307001524_0_4', 'S0022309307001524_0_5', 'S0022309307001524_0_6', 'S0022309307001524_0_7', 'S0022309307001524_0_8', 'S0022309307001524_0_9', 'S0022309307001524_0_10', 'S0022309307001524_0_11', 'S0022309307001

In [22]:
#Appending the only comp
composition_orientation = pickle.load(open(f'composition_orientation_gold.pkl', 'rb'))
for pii_tid in pred_only_comp_pii:
    inde = res_discomat['identifier'].index(pii_tid)
    pred_comp_tuples = res_discomat['pred_tuples'][inde]
    pii_tid_w = pii_tid[0] + '_' + str(pii_tid[1])
    material = {}
    for t in pred_comp_tuples:
        id = t[0]
        if id not in material:
            material[id] = []
        material[id].append(t[1:])
#     print(material)
#     break
    for key in material.keys():
        parts = key.split('_')
        my_id = ''
        if len(parts)==6:
            my_id = parts[-1]
        output_string = '_'.join(parts[:2])
        
        if composition_orientation[(parts[0], parts[1])] == 'Row oriented':
            proxy_id = pii_tid_w + '_' + 'C' + '_' + str(int(parts[3]) + 1)
        else:
            proxy_id = pii_tid_w + '_' + 'R' + '_' + str(int(parts[2]) + 1)
        
        new_list_final.append([parts[0], [int(parts[1])+1], my_id, proxy_id, material[key], ()])

In [23]:
print(len(new_list_final))
print(new_list_final[-5:])
print(new_list_final[0])
print(new_list_final[0][0])
end_search = len(new_list_final)

3659
[['S0022309302015946', [1], 'E0.73', 'S0022309302015946_0_R_3', [('Eu2O3', 0.73, 'mol'), ('MgO', 11.91, 'mol'), ('Na2O', 14.89, 'mol'), ('SiO2', 69.49, 'mol'), ('Al2O3', 2.98, 'mol')], ()], ['S0022309302015946', [1], 'E1.26', 'S0022309302015946_0_R_4', [('Eu2O3', 1.26, 'mol'), ('MgO', 11.85, 'mol'), ('Na2O', 14.81, 'mol'), ('SiO2', 69.12, 'mol'), ('Al2O3', 2.96, 'mol')], ()], ['S0022309302015946', [1], 'E3.90', 'S0022309302015946_0_R_5', [('Eu2O3', 3.9, 'mol'), ('MgO', 11.53, 'mol'), ('Na2O', 14.42, 'mol'), ('SiO2', 67.27, 'mol'), ('Al2O3', 2.88, 'mol')], ()], ['S0022309302015946', [1], 'E5.26', 'S0022309302015946_0_R_6', [('Eu2O3', 5.26, 'mol'), ('MgO', 11.37, 'mol'), ('Na2O', 14.21, 'mol'), ('SiO2', 66.32, 'mol'), ('Al2O3', 2.84, 'mol')], ()], ['S0022309302015946', [1], 'E8.11', 'S0022309302015946_0_R_7', [('Eu2O3', 8.11, 'mol'), ('MgO', 11.03, 'mol'), ('Na2O', 13.78, 'mol'), ('SiO2', 64.32, 'mol'), ('Al2O3', 2.76, 'mol')], ()]]
['S0022309309001586', [1], 'L1', 'S002230930900158

In [24]:
#Now add properties at last
for pii_tid in pred_only_prop_pii:
    inde = prop_res['test']['identifier'].index(pii_tid)
    pred_prop_tuples = result_dict[pii_tid]
    pii_tid_w = pii_tid[0] + '_' + str(pii_tid[1])
    prop_orient =prop_res['test']['ret_comp_pred'][inde]['prop_orient']
    if pii_tid == ('S0022309399003683', 1):
        print(pred_prop_tuples)
        print(prop_orient)
    #print(pred_comp_tuples)
    for tup in pred_prop_tuples:
        prop_t = (tup[1], tup[2], tup[3])
        parts = tup[0].split('_')
        my_id = ''
        if len(parts)==5:
            my_id = parts[-1]
            
        if prop_orient == 'row':
            proxy_id = pii_tid_w + '_' + 'C' + '_' + str(int(parts[3]) + 1)
        else:
            proxy_id = pii_tid_w + '_' + 'R' + '_' + str(int(parts[2]) + 1)
        
        new_list_final.append([pii_tid[0], [int(pii_tid[1])+1], my_id, proxy_id, [], prop_t])

[('S0022309399003683_1_3_1', 'Density', 2.75, 'g/cm3'), ('S0022309399003683_1_3_2', 'Density', 2.2, 'g/cm3'), ('S0022309399003683_1_3_3', 'Density', 2.54, 'g/cm3'), ('S0022309399003683_1_4_1', "Young's modulus", 98.0, 'GPa'), ('S0022309399003683_1_4_2', "Young's modulus", 72.0, 'GPa'), ('S0022309399003683_1_4_3', "Young's modulus", 73.0, 'GPa'), ('S0022309399003683_1_5_1', 'Vickers hardness', 650.0, ''), ('S0022309399003683_1_5_2', 'Vickers hardness', 500.0, ''), ('S0022309399003683_1_5_3', 'Vickers hardness', 460.0, ''), ('S0022309399003683_1_5_1', 'Vickers hardness', 650.0, 'HV'), ('S0022309399003683_1_5_2', 'Vickers hardness', 500.0, 'HV'), ('S0022309399003683_1_5_3', 'Vickers hardness', 460.0, 'HV'), ('S0022309399003683_1_1_1', 'Thermal expansion coefficient', 3.6999999999999997e-06, 'degC-1'), ('S0022309399003683_1_1_2', 'Thermal expansion coefficient', 6e-07, 'degC-1'), ('S0022309399003683_1_1_3', 'Thermal expansion coefficient', 3.76e-06, 'degC-1')]
row


In [25]:
print(len(new_list_final))

5305


In [26]:
deletion_indexes = set()
new_to_be_added = []
for comp_ind in range(start_search, end_search):
    for prop_ind in range(end_search, len(new_list_final)):
        if new_list_final[comp_ind][0] == new_list_final[prop_ind][0] and new_list_final[comp_ind][1] != new_list_final[prop_ind][1] and new_list_final[comp_ind][2] == new_list_final[prop_ind][2] and new_list_final[comp_ind][2]!='':
            new_to_be_added.append([new_list_final[comp_ind][0], new_list_final[comp_ind][1]+ new_list_final[prop_ind][1], new_list_final[comp_ind][2], '', new_list_final[comp_ind][4], new_list_final[prop_ind][5]])
            deletion_indexes.add(comp_ind)
            deletion_indexes.add(prop_ind)

In [27]:
inter_table_elem = len(new_to_be_added)
print(len(new_to_be_added))
print(new_to_be_added[0:5])
deletion_index = list(deletion_indexes)
deletion_index.sort()
print(len(deletion_index))
print(deletion_index[:5])
print(new_list_final[deletion_index[0]])

749
[['S0022309307011659', [2, 3], '21', '', [('SiO2', 62.0, 'mol'), ('B2O3', 25.0, 'mol'), ('Al2O3', 1.2, 'mol'), ('CaO', 5.4, 'mol'), ('Na2O', 3.2, 'mol'), ('Li2O', 3.2, 'mol')], ('Glass transition temperature', 500.0, 'degC')], ['S0022309307011659', [2, 3], '21', '', [('SiO2', 62.0, 'mol'), ('B2O3', 25.0, 'mol'), ('Al2O3', 1.2, 'mol'), ('CaO', 5.4, 'mol'), ('Na2O', 3.2, 'mol'), ('Li2O', 3.2, 'mol')], ('Softening Point (Temperature)', 717.0, 'degC')], ['S0022309307011659', [2, 3], '21', '', [('SiO2', 62.0, 'mol'), ('B2O3', 25.0, 'mol'), ('Al2O3', 1.2, 'mol'), ('CaO', 5.4, 'mol'), ('Na2O', 3.2, 'mol'), ('Li2O', 3.2, 'mol')], ('Thermal expansion coefficient', 5.743e-06, 'degC-1')], ['S0022309307011659', [2, 3], '22', '', [('SiO2', 62.0, 'mol'), ('B2O3', 25.0, 'mol'), ('Al2O3', 1.2, 'mol'), ('MgO', 5.4, 'mol'), ('Na2O', 3.2, 'mol'), ('Li2O', 3.2, 'mol')], ('Glass transition temperature', 480.0, 'degC')], ['S0022309307011659', [2, 3], '22', '', [('SiO2', 62.0, 'mol'), ('B2O3', 25.0, 'mol

In [28]:
comp_del, prop_del, err_del = 0,0, 0
for elem in deletion_index:
    if elem<start_search:
        err_del += 1
    elif elem<end_search:
        comp_del += 1
    else:
        prop_del += 1
        
print(err_del, comp_del, prop_del)
assert err_del==0

0 226 714


In [29]:
deletion_index.sort(reverse=True)
print(deletion_index[:5])

[5286, 5285, 5284, 5283, 5282]


In [30]:
# Sort del_list in descending order to avoid index shifting issues
len_bef_del = len(new_list_final)
print(len(new_list_final))
for index in sorted(deletion_index, reverse=True):
    del new_list_final[index]
print(len(new_list_final))
del deletion_index

5305
4365


In [31]:
new_list_final_updated = new_list_final[:start_search] + new_to_be_added + new_list_final[start_search:]

In [32]:
print(len(new_list_final_updated))
del new_list_final

5114


In [33]:
print(f'Total no of composition+prop from intra table = {start_search}')
print(f'Total no of composition+prop from inter table = {inter_table_elem}')
print(f'Total no of only composition from intra table = {end_search-start_search-comp_del}')
print(f'Total no of only properties from intra table = {len_bef_del-end_search-prop_del}')
print(f'Total no of entries in the database = {len(new_list_final_updated)}')
assert len(new_list_final_updated)== start_search + inter_table_elem + (end_search-start_search-comp_del) + (len_bef_del-end_search-prop_del)

Total no of composition+prop from intra table = 1772
Total no of composition+prop from inter table = 749
Total no of only composition from intra table = 1661
Total no of only properties from intra table = 932
Total no of entries in the database = 5114


In [34]:
deletion_indexes = []
new_to_be_added = []
for prop_ind in range(len(new_list_final_updated)-(len_bef_del-end_search-prop_del), len(new_list_final_updated)):
    for comp_ind in range(0, start_search):
        if new_list_final_updated[comp_ind][0] == new_list_final_updated[prop_ind][0] and new_list_final_updated[comp_ind][1] != new_list_final_updated[prop_ind][1] and new_list_final_updated[comp_ind][2] == new_list_final_updated[prop_ind][2] and new_list_final_updated[comp_ind][2]!='':
            new_to_be_added.append([new_list_final_updated[comp_ind][0], new_list_final_updated[comp_ind][1]+ new_list_final_updated[prop_ind][1], new_list_final_updated[comp_ind][2], '', new_list_final_updated[comp_ind][4], new_list_final_updated[prop_ind][5]])
            deletion_indexes.append(prop_ind)
            break

In [35]:
print(len(deletion_indexes))
print(len(new_to_be_added))

# Sort del_list in descending order to avoid index shifting issues
# deletion_indexes = list(deletion_indexes)
deletion_indexes.sort(reverse=True)
print(len(new_list_final_updated))

print(deletion_indexes[:5])
print(len(new_list_final_updated))

327
327
5114
[4988, 4987, 4986, 4985, 4984]
5114


In [36]:
if len(deletion_indexes)!=0:
    print('You need to write further code here')
else:
    print('Go on')

You need to write further code here


In [37]:
## Writing the further codes:

#deletion_index is reverse sorted
for index in sorted(deletion_indexes, reverse=True):
    del new_list_final_updated[index]
print(len(new_list_final_updated))
prop_del_2 = len(deletion_indexes)

del deletion_indexes

new_list_final_updated_2 = new_list_final_updated[: (start_search + inter_table_elem)] + new_to_be_added + new_list_final_updated[(start_search + inter_table_elem):]

4787


In [38]:
print(f'Total no of composition+prop from intra table = {start_search}')
print(f'Total no of composition+prop from inter table = {inter_table_elem + len(new_to_be_added)}')
print(f'Total no of only composition from intra table = {end_search-start_search-comp_del}')
print(f'Total no of only properties from intra table = {len_bef_del-end_search-prop_del-prop_del_2}')
print(f'Total no of entries in the database = {len(new_list_final_updated_2)}')
assert len(new_list_final_updated_2)== start_search + inter_table_elem + + len(new_to_be_added) + (end_search-start_search-comp_del) + (len_bef_del-end_search-prop_del-prop_del_2)

Total no of composition+prop from intra table = 1772
Total no of composition+prop from inter table = 1076
Total no of only composition from intra table = 1661
Total no of only properties from intra table = 605
Total no of entries in the database = 5114


In [39]:
ch1 = start_search
ch2 = inter_table_elem + len(new_to_be_added)
ch3 = end_search-start_search-comp_del
ch4 = len_bef_del-end_search-prop_del-prop_del_2

display(pd.DataFrame(new_list_final_updated_2[ch1-1 : ch1+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2)-1 : (ch1 + ch2)+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2 + ch3)-1 : (ch1 + ch2 + ch3)+1]))
display(pd.DataFrame(new_list_final_updated_2[(ch1 + ch2 + ch3 + ch4)-1 :]))

,0,1,2,3,4,5
0,S0022309309001410,[1],H2,S0022309309001410_0_R_7,"[(Al, 0.13, mol), (Hf, 0.62, mol), (Ni, 0.25, ...","(Bulk modulus, 128.8, GPa)"
1,S0022309307011659,"[2, 3]",21,,"[(SiO2, 62.0, mol), (B2O3, 25.0, mol), (Al2O3,...","(Glass transition temperature, 500.0, degC)"


,0,1,2,3,4,5
0,S0022309314000611,"[1, 3]",S5--Bi2O3,,"[(B2O3, 30.0, mol), (Bi2O3, 5.0, mol), (SiO2, ...","(Bulk modulus, 169.64, GPa)"
1,S0022309314003688,[2],,S0022309314003688_1_R_3,"[(O, 51.4, mol), (Ca, 5.6, mol), (B, 17.5, mol...",()


,0,1,2,3,4,5
0,S0022309302015946,[1],E8.11,S0022309302015946_0_R_7,"[(Eu2O3, 8.11, mol), (MgO, 11.03, mol), (Na2O,...",()
1,S0022309309004001,[2],1 A,S0022309309004001_1_R_2,[],"(Refractive index, 2.87, None)"


,0,1,2,3,4,5
0,S0022309305000761,[2],TeI-6,S0022309305000761_1_R_7,[],"(Poisson ratio, 0.415, None)"


In [40]:
pickle.dump(new_list_final_updated_2, open(f'database_gold.pkl', 'wb'))

In [41]:
import pandas as pd

In [42]:
column_name = ['Article PII', 'Table No.', 'ID', 'Proxy_ID', 'Composition', 'Property']
df = pd.DataFrame(new_list_final_updated_2, columns = column_name)

In [43]:
df['Journal_Name'] = 'test_data'

In [44]:
display(pd.DataFrame(df.head()))

,Article PII,Table No.,ID,Proxy_ID,Composition,Property,Journal_Name
0,S0022309309001586,[1],L1,S0022309309001586_0_R_3,"[(La, 25.0, mol), (La, 24.51, mol), (P, 75.0, ...","(Density, 3.31, g/cm3)",test_data
1,S0022309309001586,[1],L1,S0022309309001586_0_R_3,"[(La, 25.0, mol), (La, 24.51, mol), (P, 75.0, ...","(Glass transition temperature, 620.0, degC)",test_data
2,S0022309309001586,[1],C1,S0022309309001586_0_R_4,"[(Ca, 5.0, mol), (Ca, 5.26, mol), (La, 20.0, m...","(Density, 3.09, g/cm3)",test_data
3,S0022309309001586,[1],C1,S0022309309001586_0_R_4,"[(Ca, 5.0, mol), (Ca, 5.26, mol), (La, 20.0, m...","(Glass transition temperature, 562.0, degC)",test_data
4,S0022309309001586,[1],C2,S0022309309001586_0_R_5,"[(Ca, 10.0, mol), (Ca, 10.12, mol), (La, 15.0,...","(Density, 2.95, g/cm3)",test_data


In [45]:
df.to_csv(f'gold_database.csv', index=False)